In [47]:
import os
import numpy as np
import scanpy as sc
import anndata as ad
import pandas as pd
import pertpy

# Check current working directory
print(f"Current working directory: {os.getcwd()}")

Current working directory: /Users/pedroferreira/projects/DE-ZILN/notebooks/scrnaseq_reanalyses


In [48]:
adata = pertpy.data.kang_2018()
print(adata)

AnnData object with n_obs × n_vars = 24673 × 15706
    obs: 'nCount_RNA', 'nFeature_RNA', 'tsne1', 'tsne2', 'label', 'cluster', 'cell_type', 'replicate', 'nCount_SCT', 'nFeature_SCT', 'integrated_snn_res.0.4', 'seurat_clusters'
    var: 'name'
    obsm: 'X_pca', 'X_umap'


In [ ]:

# -----------------------
# Step 2: Inspect metadata
# -----------------------
# 'label' is the column that contains "ctrl" or "stim"
print("Unique condition labels in obs:", adata.obs["label"].unique())

# Rename it for clarity
adata.obs["condition"] = adata.obs["label"].astype("category")

# -----------------------------
# Step 3: Basic scanpy preprocessing
# -----------------------------
# Store raw counts for later reference
adata.layers["counts"] = adata.X.copy()

# Take only T-cells
adata = adata[adata.obs['cell_type'].str.contains('T cells')].copy()
print(adata.obs['cell_type'].value_counts())

# QC filters (these are typical thresholds; adjust if needed)
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)

# Mitochondrial content
adata.var["mt"] = adata.var_names.str.upper().str.startswith("MT-")
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], inplace=True)
adata = adata[adata.obs["pct_counts_mt"] < 20].copy()

# Remove doublets
sc.pp.scrublet(adata)
adata = adata[adata.obs["predicted_doublet"] == False].copy()

# Normalize + log1p
sc.pp.normalize_total(adata, target_sum=1e6)
sc.pp.log1p(adata)

# ------------------------
# Step 4: Run DGE (t-test)
# ------------------------
sc.tl.rank_genes_groups(
    adata,
    groupby="condition",
    groups=["stim"],
    reference="ctrl",
    method="t-test"
)

print("Top 20 DGE results:")
df = sc.get.rank_genes_groups_df(adata, group="stim")
print(df.head(20))



Unique condition labels in obs: ['ctrl', 'stim']
Categories (2, object): ['ctrl', 'stim']
cell_type
CD4 T cells    11238
CD8 T cells     1621
Name: count, dtype: int64
Top 20 DGE results:
      names      scores  logfoldchanges  pvals  pvals_adj
0     ISG15  192.520645       10.905020    0.0        0.0
1      IFI6  179.198242       11.142301    0.0        0.0
2     IFIT3  152.896469       11.938686    0.0        0.0
3     IFIT1  141.387939       11.451269    0.0        0.0
4       MX1  113.908150        8.966833    0.0        0.0
5      LY6E  112.923683        8.427493    0.0        0.0
6     ISG20  100.401672        7.316315    0.0        0.0
7     IFIT2   87.951065        9.176761    0.0        0.0
8      OAS1   83.281662        8.123415    0.0        0.0
9      MT2A   77.013077        7.083405    0.0        0.0
10   IFI44L   72.969772        7.479745    0.0        0.0
11    RSAD2   72.616829        8.724595    0.0        0.0
12     IRF7   66.330666        6.037930    0.0        0.0


In [ ]:
# -------------------------------
# Negative control: Split control condition into two random groups
# -------------------------------
# Filter to only control cells
ctrl_adata = adata[adata.obs["condition"] == "ctrl"].copy()
ctrl_adata.X = ctrl_adata.layers['counts'].copy()
sc.pp.filter_genes(ctrl_adata, min_cells=3)
sc.pp.normalize_total(ctrl_adata, target_sum=1e6)
sc.pp.log1p(ctrl_adata)

print(f"Total control cells: {ctrl_adata.n_obs}")

rng = np.random.default_rng(42)

reps = np.array(sorted(ctrl_adata.obs["replicate"].unique()))
rng.shuffle(reps)

# Split replicates roughly in half
half = len(reps) // 2
groupA = set(reps[:half])
groupB = set(reps[half:])

# Assign each cell based on replicate membership
grp = ctrl_adata.obs["replicate"].map(lambda r: "A" if r in groupA else "B")
ctrl_adata.obs["rand_group"] = grp.astype("category")

# Sanity check: replicate counts per group
rep_counts = ctrl_adata.obs.drop_duplicates("replicate")["rand_group"].value_counts()
print("Replicates per random group:", rep_counts.to_dict())

# Cell counts per group
print("Cells per random group:", ctrl_adata.obs["rand_group"].value_counts().to_dict())

# Run cell-level DE ignoring replicate structure (this is the point)
sc.tl.rank_genes_groups(
    ctrl_adata,
    groupby="rand_group",
    groups=["A"],
    reference="B",
    method='t-test',
    use_raw=False
)
results_df = sc.get.rank_genes_groups_df(ctrl_adata, group="A")


# Count significant genes at different thresholds
alpha = 0.05
n_de_05 = (results_df["pvals_adj"] < alpha).sum()
n_de_01 = (results_df["pvals_adj"] < 0.01).sum()
n_de_001 = (results_df["pvals_adj"] < 0.001).sum()

print("\n" + "="*60)
print("Negative Control Results (Control vs Control)")
print("="*60)
print(f"Total genes tested: {len(results_df)}")
print(f"DE genes (FDR < 0.05): {n_de_05}")
print(f"DE genes (FDR < 0.01): {n_de_01}")
print(f"DE genes (FDR < 0.001): {n_de_001}")
print(f"\nFalse positive rate (FDR < 0.05): {n_de_05 / len(results_df) * 100:.2f}%")

print("\nTop 10 'differentially expressed' genes (should be false positives):")
print(results_df.head(10)[["names", "pvals", "pvals_adj", "scores", "logfoldchanges"]])


Total control cells: 6371
Replicates per random group: {'A': 4, 'B': 4}
Cells per random group: {'B': 3718, 'A': 2653}

Negative Control Results (Control vs Control)
Total genes tested: 12579
DE genes (FDR < 0.05): 270
DE genes (FDR < 0.01): 158
DE genes (FDR < 0.001): 96

False positive rate (FDR < 0.05): 2.15%

Top 10 'differentially expressed' genes (should be false positives):
      names         pvals     pvals_adj     scores  logfoldchanges
0      GZMB  9.020368e-60  5.673360e-56  16.597183        2.836433
1      GZMH  7.737341e-48  2.433200e-44  14.735089        2.579262
2    FGFBP2  1.212691e-30  3.050888e-27  11.608372        2.231050
3      NKG7  1.934638e-30  4.055969e-27  11.549190        1.766002
4      GNLY  3.426583e-29  6.157570e-26  11.297683        1.905162
5      CCL4  2.629617e-24  3.675328e-21  10.240518        2.000736
6      RPS7  8.192219e-16  7.926917e-13   8.072721        0.590540
7      CST7  1.640244e-15  1.473759e-12   7.992194        1.184024
8  APOBEC3G  

In [37]:
# Run cell-level DE ignoring replicate structure (this is the point)
sc.tl.rank_genes_groups(
    ctrl_adata,
    groupby="rand_group",
    groups=["A"],
    reference="B",
    method='wilcoxon',
    use_raw=False
)
results_df = sc.get.rank_genes_groups_df(ctrl_adata, group="A")


# Count significant genes at different thresholds
alpha = 0.05
n_de_05 = (results_df["pvals_adj"] < alpha).sum()
n_de_01 = (results_df["pvals_adj"] < 0.01).sum()
n_de_001 = (results_df["pvals_adj"] < 0.001).sum()

print("\n" + "="*60)
print("Negative Control Results (Control vs Control)")
print("="*60)
print(f"Total genes tested: {len(results_df)}")
print(f"DE genes (FDR < 0.05): {n_de_05}")
print(f"DE genes (FDR < 0.01): {n_de_01}")
print(f"DE genes (FDR < 0.001): {n_de_001}")
print(f"\nFalse positive rate (FDR < 0.05): {n_de_05 / len(results_df) * 100:.2f}%")

print("\nTop 10 'differentially expressed' genes (should be false positives):")
print(results_df.head(10)[["names", "pvals", "pvals_adj", "scores", "logfoldchanges"]])


Negative Control Results (Control vs Control)
Total genes tested: 12579
DE genes (FDR < 0.05): 46
DE genes (FDR < 0.01): 34
DE genes (FDR < 0.001): 28

False positive rate (FDR < 0.05): 0.37%

Top 10 'differentially expressed' genes (should be false positives):
   names         pvals     pvals_adj     scores  logfoldchanges
0  HLA-B  1.890031e-68  1.188735e-64  17.484213        0.329520
1   RPS7  3.432911e-40  1.439419e-36  13.270498        0.590540
2   GZMB  7.131743e-22  2.242755e-18   9.611763        2.836433
3   GZMH  1.528197e-15  2.746169e-12   7.974645        2.579262
4   NKG7  3.838319e-14  6.035277e-11   7.566359        1.766002
5   RPL6  1.191591e-11  1.362638e-08   6.781228        0.178017
6  ANXA1  1.443966e-10  1.397203e-07   6.411181        0.861520
7   GNLY  2.673919e-10  2.402516e-07   6.316593        1.905162
8   CCL5  2.489452e-09  1.957176e-06   5.962147        1.178230
9   RPS8  2.287922e-08  1.514725e-05   5.588687        0.075557


In [51]:
import importlib
import os

file_path = os.path.join(os.getcwd(), "../../pkg/", "scanpy_wrapper.py")
spec = importlib.util.spec_from_file_location("scanpy_wrapper", file_path)
scanpy_wrapper = importlib.util.module_from_spec(spec)
spec.loader.exec_module(scanpy_wrapper)

# LN's t-test uses normalized data without log1p
ctrl_adata_norm = ctrl_adata.copy()
ctrl_adata_norm.X = ctrl_adata_norm.layers['counts'].copy()
sc.pp.normalize_total(ctrl_adata_norm, target_sum=1e6)
scanpy_wrapper.rank_genes_groups_ln(
    ctrl_adata_norm,
    groupby="rand_group",
    groups=["A"],
    reference="B",
    use_raw=False
)

Processing group: A


In [52]:
results_df = sc.get.rank_genes_groups_df(ctrl_adata_norm, group="A")


# Count significant genes at different thresholds
alpha = 0.05
n_de_05 = (results_df["pvals_adj"] < alpha).sum()
n_de_01 = (results_df["pvals_adj"] < 0.01).sum()
n_de_001 = (results_df["pvals_adj"] < 0.001).sum()

print("\n" + "="*60)
print("Negative Control Results (Control vs Control)")
print("="*60)
print(f"Total genes tested: {len(results_df)}")
print(f"DE genes (FDR < 0.05): {n_de_05}")
print(f"DE genes (FDR < 0.01): {n_de_01}")
print(f"DE genes (FDR < 0.001): {n_de_001}")
print(f"\nFalse positive rate (FDR < 0.05): {n_de_05 / len(results_df) * 100:.2f}%")

print("\nTop 10 'differentially expressed' genes (should be false positives):")
print(results_df.head(10)[["names", "pvals", "pvals_adj", "scores", "logfoldchanges"]])


Negative Control Results (Control vs Control)
Total genes tested: 12579
DE genes (FDR < 0.05): 73
DE genes (FDR < 0.01): 56
DE genes (FDR < 0.001): 43

False positive rate (FDR < 0.05): 0.58%

Top 10 'differentially expressed' genes (should be false positives):
      names         pvals     pvals_adj     scores  logfoldchanges
0     HLA-B  0.000000e+00  0.000000e+00  17.475964        0.372301
1      GZMB  0.000000e+00  4.189882e-43  14.566743        2.249869
2      RPS7  3.274835e-42  4.119745e-38  13.722983        0.323870
3      GZMH  2.190687e-36  2.755665e-32  12.685308        2.019061
4    FGFBP2  9.772758e-25  1.229315e-20  10.315315        1.980004
5      CCL4  2.232621e-23  2.808414e-19  10.002950        1.844573
6      CCL5  4.333991e-23  5.451727e-19   9.934540        0.938112
7  APOBEC3G  2.058795e-22  2.589758e-18   9.776210        0.967968
8      NKG7  5.385884e-22  6.774903e-18   9.677068        1.072999
9      CST7  3.175151e-20  3.994023e-16   9.244304        1.002979
